# Nested Learning | Frameworks & Meta-Approaches

In [1]:
# Nested Learning: 3 loops where each level's output changes the next iteration's behavior
import re
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# Mutable agent state — each loop modifies this
base_prompt = "Answer the question."
prompt_mods: List[str] = []           # accumulated modifiers (inner/middle loop)
tools: List[str] = ["basic_lookup"]   # available tools (outer loop)
cot_help_count = 0                    # tracks how often CoT helped (middle loop)
cot_total = 0

questions = [
    "What is 15% of 260?", "What is the capital of Bhutan?",
    "If a car goes 80km/h for 2.5 hours, how far does it travel?",
    "What year was the Treaty of Westphalia signed?",
    "What is 7 factorial?", "Name the largest ocean.",
]

def build_prompt(question: str) -> str:
    """Assemble the current prompt from base + modifications + tool list."""
    mods = " ".join(prompt_mods) if prompt_mods else ""
    tool_note = f"You have these tools: {', '.join(tools)}. Use them if helpful." if tools else ""
    return f"{base_prompt} {mods}\n{tool_note}\nQuestion: {question}"

def score_answer(question: str, answer: str) -> float:
    """LLM scores answer quality 0.0-1.0."""
    resp = model.invoke(
        f"Question: {question}\nAnswer: {answer}\n\n"
        "Rate the answer's correctness 0.0-1.0. Reply with ONLY a number."
    )
    try: return max(0.0, min(1.0, float(re.search(r"[\d.]+", resp.content).group())))
    except: return 0.5

In [4]:
# --- Inner loop: per-task retry with prompt modification ---
def inner_loop(question: str) -> dict:
    global cot_help_count, cot_total
    prompt = build_prompt(question)
    answer = model.invoke(prompt).content.strip()
    score = score_answer(question, answer)
    used_cot = False
    if score < 0.7:  # retry with chain-of-thought
        cot_prompt = build_prompt(question) + "\nThink step by step before answering."
        answer2 = model.invoke(cot_prompt).content.strip()
        score2 = score_answer(question, answer2)
        if score2 > score:
            cot_help_count += 1; used_cot = True
            answer, score = answer2, score2
        cot_total += 1
    return {"question": question, "score": score, "used_cot": used_cot}

# --- Middle loop: after a batch, decide if CoT should be permanent ---
def middle_loop(batch_results: List[dict]):
    global cot_help_count, cot_total
    avg = sum(r["score"] for r in batch_results) / len(batch_results)
    if cot_total > 0 and cot_help_count / cot_total > 0.6:
        if "Think step by step." not in prompt_mods:
            prompt_mods.append("Think step by step.")
            print(f"  [Middle] CoT helped {cot_help_count}/{cot_total} times -> made permanent")
    cot_help_count, cot_total = 0, 0  # reset for next batch
    return avg

# --- Outer loop: if performance plateaus, add calculator tool ---
def outer_loop(epoch_scores: List[float]):
    if len(epoch_scores) >= 2:
        recent = epoch_scores[-2:]
        if all(s < 0.8 for s in recent) and "calculator" not in tools:
            tools.append("calculator")
            prompt_mods.append("For math questions, use the calculator tool.")
            print(f"  [Outer] Performance plateaued -> added calculator tool")

In [5]:
# --- Run: 2 epochs x 3 questions per batch ---
epoch_scores = []
for epoch in range(2):
    batch = questions[epoch * 3:(epoch + 1) * 3]
    results = [inner_loop(q) for q in batch]
    for r in results:
        print(f"  Q: {r['question'][:40]}  score={r['score']:.2f}  cot={r['used_cot']}")
    avg = middle_loop(results)
    epoch_scores.append(avg)
    print(f"Epoch {epoch+1} avg: {avg:.2f} | mods: {prompt_mods} | tools: {tools}\n")
    outer_loop(epoch_scores)

print(f"Final prompt mods: {prompt_mods}")
print(f"Final tools: {tools}")
print(f"Epoch scores: {[f'{s:.2f}' for s in epoch_scores]}")
print(f"\nFinal prompt modifications: {len(prompt_mods)} | Tools added: {len(tools) - 1}")

  Q: What is 15% of 260?  score=1.00  cot=False
  Q: What is the capital of Bhutan?  score=1.00  cot=False
  Q: If a car goes 80km/h for 2.5 hours, how   score=1.00  cot=False
Epoch 1 avg: 1.00 | mods: [] | tools: ['basic_lookup']

  Q: What year was the Treaty of Westphalia s  score=1.00  cot=False
  Q: What is 7 factorial?  score=1.00  cot=False
  Q: Name the largest ocean.  score=1.00  cot=False
Epoch 2 avg: 1.00 | mods: [] | tools: ['basic_lookup']

Final prompt mods: []
Final tools: ['basic_lookup']
Epoch scores: ['1.00', '1.00']

Final prompt modifications: 0 | Tools added: 0
